# 🏦 EDA — Home Credit Default Risk
## Análisis Exploratorio de Datos | SparkSQL + PySpark + AWS Glue/S3

**Proyecto Integrador — MCDA 2026-1 | Universidad EAFIT**  
**Curso:** SI7006 — Almacenamiento y Procesamiento de Grandes Datos

---

### Objetivo
Este notebook reemplaza el EDA anterior y mantiene el mismo alcance, pero ahora está orientado a cumplir explícitamente el requisito de:

> Realizar Análisis Exploratorio de Datos con SQL/SparkSQL utilizando como entrada tablas catalogadas.

El notebook permite:

- Leer tablas desde **AWS Glue Data Catalog** cuando estén disponibles.
- Usar **SparkSQL** para el EDA principal.
- Usar PySpark para apoyo técnico: carga, escritura, conversión a pandas para gráficos y guardado de resultados.
- Generar evidencias para el informe.
- Producir un dataset maestro preparado para Track C / Track D.

---

### Flujo esperado

```text
S3 raw/bronze
↓
Glue Data Catalog: raw_db
↓
SparkSQL EDA
↓
Decisiones de limpieza
↓
Features y agregaciones
↓
S3 trusted/gold
↓
S3 refined/silver para resultados del EDA
```


---
## ⚙️ SECCIÓN 0 — Configuración del entorno


In [ ]:
# ============================================================
# CELDA 0.1 — Importaciones globales
# ============================================================

import os
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, NumericType

print("✅ Librerías importadas correctamente")


In [ ]:

## ⚙️ SECCIÓN 0 — Configuración del entorno


# Cambia este valor por el nombre real de tu bucket
BUCKET_NAME = "hcdr"  # Ejemplo: "hcdr-proyecto-eafit-2026"

RAW_PATH     = "s3://{}/raw/".format(BUCKET_NAME)
TRUSTED_PATH = "s3://{}/trusted/".format(BUCKET_NAME)
REFINED_PATH = "s3://{}/refined/eda/".format(BUCKET_NAME)

# Bases de datos esperadas en Glue Catalog
RAW_DB = "raw_db"
TRUSTED_DB = "hcdr_trusted"

# Si ya tienes tablas catalogadas en Glue, deja esto en True.
# Si estás probando local/temporalmente desde CSV, ponlo en False.
USE_GLUE_CATALOG = True

# Si alguna tabla catalogada no existe, el notebook puede usar S3 como fallback.
ALLOW_S3_FALLBACK = False

# Fracción para graficar variables numéricas. Evita hacer toPandas() de toda la tabla.
SAMPLE_FRACTION = 0.03

PATHS = {
    "application_train"      : RAW_PATH + "application_train/application_train.csv",
    "bureau"                 : RAW_PATH + "bureau/bureau.csv",
    "bureau_balance"         : RAW_PATH + "bureau_balance/bureau_balance.csv",
    "previous_application"   : RAW_PATH + "previous_application/previous_application.csv",
    "pos_cash_balance"       : RAW_PATH + "pos_cash_balance/POS_CASH_balance.csv",
    "credit_card_balance"    : RAW_PATH + "credit_card_balance/credit_card_balance.csv",
    "installments_payments"  : RAW_PATH + "installments_payments/installments_payments.csv",
}

print("✅ Configuración lista")
print("RAW_PATH     :", RAW_PATH)
print("TRUSTED_PATH :", TRUSTED_PATH)
print("REFINED_PATH :", REFINED_PATH)
print("RAW_DB       :", RAW_DB)
print("USE_GLUE_CATALOG:", USE_GLUE_CATALOG)


In [ ]:
# ============================================================
# CELDA 0.3 — Inicialización de Spark / Glue
# ============================================================

try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext

    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print("✅ Spark inicializado desde AWS GlueContext")
except Exception as e:
    print("⚠️ No se pudo usar GlueContext. Se usará SparkSession estándar.")
    spark = (
        SparkSession.builder
        .appName("EDA_HomeCreditRisk_SparkSQL_AWS")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.shuffle.partitions", "120")
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


In [ ]:
# ============================================================
# CELDA 0.4 — Funciones auxiliares para SQL, tablas y gráficos
# ============================================================

def normalize_column_names(df):
    """Convierte nombres de columnas a minúsculas para evitar problemas entre CSV, Glue y Athena."""
    for old in df.columns:
        new = old.strip().lower().replace(" ", "_")
        if old != new:
            df = df.withColumnRenamed(old, new)
    return df


def load_table(view_name, catalog_table=None, s3_path=None):
    """Carga una tabla desde Glue Catalog si es posible; si falla, usa CSV desde S3 como fallback."""
    catalog_table = catalog_table or view_name

    if USE_GLUE_CATALOG:
        try:
            full_name = "{}.{}".format(RAW_DB, catalog_table)
            print("📚 Leyendo tabla catalogada:", full_name)
            df = spark.table(full_name)
            df = normalize_column_names(df)
            df.createOrReplaceTempView(view_name)
            print("   ✅ Vista SparkSQL creada: {} | {:,} filas × {} columnas".format(view_name, df.count(), len(df.columns)))
            return df
        except Exception as e:
            print("⚠️ No se pudo leer desde Glue Catalog:", catalog_table)
            print("   Motivo:", str(e)[:250])
            if not ALLOW_S3_FALLBACK:
                raise e

    if s3_path is None:
        raise ValueError("No hay ruta S3 para fallback de {}".format(view_name))

    print("📥 Leyendo CSV desde S3:", s3_path)
    df = spark.read.csv(s3_path, header=True, inferSchema=True, nullValue="", nanValue="NaN")
    df = normalize_column_names(df)
    df.createOrReplaceTempView(view_name)
    print("   ✅ Vista SparkSQL creada: {} | {:,} filas × {} columnas".format(view_name, df.count(), len(df.columns)))
    return df


def sql(query, rows=20, truncate=False):
    """Ejecuta SparkSQL y muestra resultados."""
    result = spark.sql(query)
    result.show(rows, truncate=truncate)
    return result


def guardar_spark_en_refined(df, nombre):
    """Guarda un Spark DataFrame en refined/silver como CSV con header."""
    out_path = REFINED_PATH + nombre.strip("/") + "/"
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", "true")
          .csv(out_path)
    )
    print("💾 Guardado en refined/silver:", out_path)


def guardar_parquet_trusted(df, nombre):
    """Guarda un Spark DataFrame en trusted/gold como Parquet."""
    out_path = TRUSTED_PATH + nombre.strip("/") + "/"
    (
        df.write
          .mode("overwrite")
          .option("compression", "snappy")
          .parquet(out_path)
    )
    print("💾 Guardado en trusted/gold:", out_path)


def plot_bar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None, rotation=0):
    if pdf.empty:
        print("⚠️ DataFrame vacío. No se genera gráfico.")
        return
    plt.figure(figsize=(10, 4))
    plt.bar(pdf[x].astype(str), pdf[y])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha="right" if rotation else "center")
    plt.tight_layout()
    plt.show()


def plot_hbar_from_pandas(pdf, x, y, title, xlabel=None, ylabel=None):
    if pdf.empty:
        print("⚠️ DataFrame vacío. No se genera gráfico.")
        return
    plt.figure(figsize=(10, max(4, len(pdf) * 0.35)))
    plt.barh(pdf[y].astype(str), pdf[x])
    plt.title(title)
    plt.xlabel(xlabel or x)
    plt.ylabel(ylabel or y)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


def build_null_sql(view_name, columns):
    """Construye SQL para reporte vertical de nulos por columna."""
    selects = []
    stack_entries = []
    for i, c in enumerate(columns):
        alias = "n_{}".format(i)
        selects.append("SUM(CASE WHEN `{}` IS NULL THEN 1 ELSE 0 END) AS {}".format(c, alias))
        stack_entries.append("'{}', {}".format(c, alias))

    query = """
    WITH base AS (
        SELECT
            COUNT(*) AS total,
            {selects}
        FROM {view_name}
    )
    SELECT
        columna,
        nulos,
        ROUND(100.0 * nulos / total, 2) AS pct_nulos
    FROM base
    LATERAL VIEW stack({n}, {stack_entries}) s AS columna, nulos
    ORDER BY pct_nulos DESC, nulos DESC
    """.format(
        selects=",\n            ".join(selects),
        view_name=view_name,
        n=len(columns),
        stack_entries=", ".join(stack_entries)
    )
    return query


def value_counts_sql(view_name, column_name, target_col="target"):
    """Distribución y tasa de default para una columna categórica."""
    return """
    SELECT
        '{col}' AS variable,
        CAST(`{col}` AS STRING) AS valor,
        COUNT(*) AS total,
        SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) AS defaults,
        ROUND(100.0 * SUM(CASE WHEN `{target}` = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
    FROM {view}
    WHERE `{col}` IS NOT NULL
      AND `{target}` IN (0, 1)
    GROUP BY `{col}`
    ORDER BY total DESC
    """.format(col=column_name, target=target_col, view=view_name)

print("✅ Funciones auxiliares definidas")


---
## 📥 SECCIÓN 1 — Carga de datos desde Glue Catalog / S3 raw


In [ ]:
# ============================================================
# CELDA 1.1 — Carga de todas las tablas como vistas SparkSQL
# ============================================================

TABLES = {}

for name, path in PATHS.items():
    TABLES[name] = load_table(name, catalog_table=name, s3_path=path)

# DataFrames principales
app = TABLES["application_train"]
bureau = TABLES["bureau"]
bureau_balance = TABLES["bureau_balance"]
previous_application = TABLES["previous_application"]
pos_cash_balance = TABLES["pos_cash_balance"]
credit_card_balance = TABLES["credit_card_balance"]
installments_payments = TABLES["installments_payments"]

print("\n✅ Todas las tablas quedaron disponibles como vistas SparkSQL")
spark.sql("SHOW TABLES").show(truncate=False)


In [ ]:
# ============================================================
# CELDA 1.2 — Inventario general del datalake raw con SparkSQL
# ============================================================

inventory_rows = []

for nombre, df in TABLES.items():
    temp_view = "tmp_inv_{}".format(nombre)
    spark.sql("""
        SELECT
            '{nombre}' AS tabla,
            COUNT(*) AS filas
        FROM {nombre}
    """.format(nombre=nombre)).createOrReplaceTempView(temp_view)

inventory_query = " UNION ALL ".join([
    "SELECT * FROM tmp_inv_{}".format(nombre) for nombre in TABLES.keys()
])

inventario_sql = spark.sql(inventory_query)
inventario_sql.show(truncate=False)
guardar_spark_en_refined(inventario_sql, "inventario_raw")


In [ ]:
# ============================================================
# CELDA 1.3 — Vista de columnas disponibles por tabla
# ============================================================

schema_rows = []
for nombre, df in TABLES.items():
    for field in df.schema.fields:
        schema_rows.append((nombre, field.name, str(field.dataType)))

schema_df = spark.createDataFrame(schema_rows, ["tabla", "columna", "tipo"])
schema_df.createOrReplaceTempView("schema_inventario")

sql("""
SELECT tabla, columna, tipo
FROM schema_inventario
ORDER BY tabla, columna
""", rows=200, truncate=False)

guardar_spark_en_refined(schema_df, "schema_inventario")


---
## 🔬 SECCIÓN 2 — EDA con SparkSQL sobre `application_train`


In [ ]:
# ============================================================
# CELDA 2.1 — Estructura básica de application_train
# ============================================================

sql("""
SELECT COUNT(*) AS total_filas
FROM application_train
""")

sql("""
SELECT *
FROM application_train
LIMIT 10
""", rows=10, truncate=False)


In [ ]:
# ============================================================
# CELDA 2.2 — Variable objetivo TARGET
# ============================================================

target_dist = sql("""
SELECT
    target,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM application_train
WHERE target IN (0, 1)
GROUP BY target
ORDER BY target
""")

guardar_spark_en_refined(target_dist, "target_distribution")

target_pd = target_dist.toPandas()
plot_bar_from_pandas(
    target_pd,
    x="target",
    y="total",
    title="Distribución de TARGET",
    xlabel="TARGET",
    ylabel="Cantidad"
)


In [ ]:
# ============================================================
# CELDA 2.3 — Análisis de nulos en application_train con SparkSQL
# ============================================================

nulls_app = spark.sql(build_null_sql("application_train", app.columns))
nulls_app.show(50, truncate=False)

guardar_spark_en_refined(nulls_app, "nulos_application_train")

nulls_app_pd = nulls_app.limit(30).toPandas()
plot_hbar_from_pandas(
    nulls_app_pd,
    x="pct_nulos",
    y="columna",
    title="Top 30 columnas con mayor porcentaje de nulos — application_train",
    xlabel="% de nulos",
    ylabel="Columna"
)


In [ ]:
# ============================================================
# CELDA 2.4 — Estadísticas descriptivas numéricas con SparkSQL
# ============================================================

num_summary = sql("""
SELECT 'amt_income_total' AS variable,
       COUNT(amt_income_total) AS n,
       ROUND(MIN(amt_income_total), 2) AS min,
       ROUND(percentile_approx(amt_income_total, 0.25), 2) AS p25,
       ROUND(percentile_approx(amt_income_total, 0.50), 2) AS mediana,
       ROUND(percentile_approx(amt_income_total, 0.75), 2) AS p75,
       ROUND(AVG(amt_income_total), 2) AS promedio,
       ROUND(MAX(amt_income_total), 2) AS max
FROM application_train
UNION ALL
SELECT 'amt_credit', COUNT(amt_credit), ROUND(MIN(amt_credit),2), ROUND(percentile_approx(amt_credit,0.25),2), ROUND(percentile_approx(amt_credit,0.50),2), ROUND(percentile_approx(amt_credit,0.75),2), ROUND(AVG(amt_credit),2), ROUND(MAX(amt_credit),2)
FROM application_train
UNION ALL
SELECT 'amt_annuity', COUNT(amt_annuity), ROUND(MIN(amt_annuity),2), ROUND(percentile_approx(amt_annuity,0.25),2), ROUND(percentile_approx(amt_annuity,0.50),2), ROUND(percentile_approx(amt_annuity,0.75),2), ROUND(AVG(amt_annuity),2), ROUND(MAX(amt_annuity),2)
FROM application_train
UNION ALL
SELECT 'amt_goods_price', COUNT(amt_goods_price), ROUND(MIN(amt_goods_price),2), ROUND(percentile_approx(amt_goods_price,0.25),2), ROUND(percentile_approx(amt_goods_price,0.50),2), ROUND(percentile_approx(amt_goods_price,0.75),2), ROUND(AVG(amt_goods_price),2), ROUND(MAX(amt_goods_price),2)
FROM application_train
UNION ALL
SELECT 'days_birth', COUNT(days_birth), ROUND(MIN(days_birth),2), ROUND(percentile_approx(days_birth,0.25),2), ROUND(percentile_approx(days_birth,0.50),2), ROUND(percentile_approx(days_birth,0.75),2), ROUND(AVG(days_birth),2), ROUND(MAX(days_birth),2)
FROM application_train
UNION ALL
SELECT 'days_employed', COUNT(days_employed), ROUND(MIN(days_employed),2), ROUND(percentile_approx(days_employed,0.25),2), ROUND(percentile_approx(days_employed,0.50),2), ROUND(percentile_approx(days_employed,0.75),2), ROUND(AVG(days_employed),2), ROUND(MAX(days_employed),2)
FROM application_train
""", rows=50, truncate=False)

guardar_spark_en_refined(num_summary, "estadisticas_numericas_application_train")


In [ ]:
# ============================================================
# CELDA 2.5 — Variables categóricas clave con SparkSQL
# ============================================================

categoricas_clave = [
    "code_gender",
    "name_contract_type",
    "name_income_type",
    "name_education_type",
    "name_family_status",
    "name_housing_type",
    "occupation_type",
    "organization_type"
]

cat_results = []
for c in categoricas_clave:
    print("\n=====", c, "=====")
    r = spark.sql(value_counts_sql("application_train", c))
    r.show(30, truncate=False)
    cat_results.append(r)

cat_union = cat_results[0]
for r in cat_results[1:]:
    cat_union = cat_union.unionByName(r)

guardar_spark_en_refined(cat_union, "categoricas_application_train")


In [ ]:
# ============================================================
# CELDA 2.6 — Variables financieras clave: histogramas con muestra
# ============================================================

money_sample = spark.sql("""
SELECT
    amt_income_total,
    amt_credit,
    amt_annuity,
    amt_goods_price
FROM application_train
WHERE target IN (0, 1)
""").sample(False, SAMPLE_FRACTION, seed=42).toPandas()

for c in ["amt_income_total", "amt_credit", "amt_annuity", "amt_goods_price"]:
    if c in money_sample.columns:
        plt.figure(figsize=(8, 4))
        money_sample[c].dropna().hist(bins=50)
        plt.title("Distribución de {}".format(c))
        plt.xlabel(c)
        plt.ylabel("Frecuencia")
        plt.tight_layout()
        plt.show()


In [ ]:
# ============================================================
# CELDA 2.7 — Variables demográficas: edad e ingresos
# ============================================================

age_income = sql("""
SELECT
    target,
    COUNT(*) AS total,
    ROUND(AVG(ABS(days_birth) / 365.0), 2) AS avg_age_years,
    ROUND(percentile_approx(ABS(days_birth) / 365.0, 0.50), 2) AS median_age_years,
    ROUND(AVG(amt_income_total), 2) AS avg_income,
    ROUND(percentile_approx(amt_income_total, 0.50), 2) AS median_income
FROM application_train
WHERE target IN (0, 1)
GROUP BY target
ORDER BY target
""", rows=10, truncate=False)

guardar_spark_en_refined(age_income, "edad_ingreso_por_target")

age_pd = spark.sql("""
SELECT ABS(days_birth) / 365.0 AS age_years
FROM application_train
WHERE days_birth IS NOT NULL
""").sample(False, SAMPLE_FRACTION, seed=42).toPandas()

plt.figure(figsize=(8, 4))
age_pd["age_years"].dropna().hist(bins=40)
plt.title("Distribución de edad aproximada")
plt.xlabel("Edad en años")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELDA 2.8 — Scores externos EXT_SOURCE_1/2/3
# ============================================================

ext_summary = sql("""
SELECT 'ext_source_1' AS variable,
       COUNT(ext_source_1) AS n,
       ROUND(AVG(ext_source_1), 4) AS promedio,
       ROUND(percentile_approx(ext_source_1, 0.50), 4) AS mediana,
       ROUND(corr(CAST(target AS DOUBLE), ext_source_1), 4) AS corr_target
FROM application_train
WHERE target IN (0, 1)
UNION ALL
SELECT 'ext_source_2', COUNT(ext_source_2), ROUND(AVG(ext_source_2),4), ROUND(percentile_approx(ext_source_2,0.50),4), ROUND(corr(CAST(target AS DOUBLE), ext_source_2),4)
FROM application_train
WHERE target IN (0, 1)
UNION ALL
SELECT 'ext_source_3', COUNT(ext_source_3), ROUND(AVG(ext_source_3),4), ROUND(percentile_approx(ext_source_3,0.50),4), ROUND(corr(CAST(target AS DOUBLE), ext_source_3),4)
FROM application_train
WHERE target IN (0, 1)
""", rows=10, truncate=False)

guardar_spark_en_refined(ext_summary, "ext_source_summary")


In [ ]:
# ============================================================
# CELDA 2.9 — Detección de outliers con IQR usando SparkSQL
# ============================================================

outlier_dfs = []

columnas_outliers = [
    "amt_income_total",
    "amt_credit",
    "amt_annuity",
    "amt_goods_price"
]

for c in columnas_outliers:

    query = f"""
    WITH q AS (
        SELECT percentile_approx({c}, array(0.25, 0.75)) AS qs
        FROM application_train
        WHERE {c} IS NOT NULL
    ),
    bounds AS (
        SELECT
            element_at(qs, 1) AS q1,
            element_at(qs, 2) AS q3,
            element_at(qs, 2) - element_at(qs, 1) AS iqr
        FROM q
    )
    SELECT
        '{c}' AS variable,
        q1,
        q3,
        iqr,
        q1 - 1.5 * iqr AS lower_bound,
        q3 + 1.5 * iqr AS upper_bound,
        SUM(
            CASE
                WHEN a.{c} < q1 - 1.5 * iqr
                  OR a.{c} > q3 + 1.5 * iqr
                THEN 1
                ELSE 0
            END
        ) AS outliers,
        COUNT(*) AS total,
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN a.{c} < q1 - 1.5 * iqr
                      OR a.{c} > q3 + 1.5 * iqr
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS pct_outliers
    FROM application_train a
    CROSS JOIN bounds
    WHERE a.{c} IS NOT NULL
    GROUP BY q1, q3, iqr
    """

    outlier_df = spark.sql(query)
    outlier_dfs.append(outlier_df)


# Unir todos los resultados
outliers_sql = outlier_dfs[0]

for df_tmp in outlier_dfs[1:]:
    outliers_sql = outliers_sql.unionByName(df_tmp)


outliers_sql.show(truncate=False)

guardar_spark_en_refined(
    outliers_sql,
    "outliers_iqr_application_train"
)

In [ ]:
# ============================================================
# CELDA 2.10 — Correlación entre variables numéricas y TARGET
# ============================================================

corr_variables = [
    "amt_income_total", "amt_credit", "amt_annuity", "amt_goods_price",
    "days_birth", "days_employed", "days_registration", "days_id_publish",
    "region_population_relative", "cnt_children", "cnt_fam_members",
    "ext_source_1", "ext_source_2", "ext_source_3"
]

corr_parts = []
for c in corr_variables:
    if c in app.columns:
        corr_parts.append("""
        SELECT '{c}' AS variable,
               ROUND(corr(CAST(target AS DOUBLE), CAST(`{c}` AS DOUBLE)), 6) AS corr_target
        FROM application_train
        WHERE target IN (0, 1) AND `{c}` IS NOT NULL
        """.format(c=c))

corr_sql = spark.sql(" UNION ALL ".join(corr_parts))
corr_sql.createOrReplaceTempView("corr_target")

corr_sorted = sql("""
SELECT variable, corr_target, ABS(corr_target) AS abs_corr
FROM corr_target
ORDER BY abs_corr DESC
""", rows=50, truncate=False)

guardar_spark_en_refined(corr_sorted, "correlaciones_target")

corr_pd = corr_sorted.limit(20).toPandas()
plot_hbar_from_pandas(
    corr_pd,
    x="corr_target",
    y="variable",
    title="Top 20 correlaciones con TARGET",
    xlabel="Correlación",
    ylabel="Variable"
)


In [ ]:
# ============================================================
# CELDA 2.11 — Tasas de incumplimiento por categorías
# ============================================================

for c in ["code_gender", "name_contract_type", "name_education_type", "name_income_type", "name_family_status"]:
    print("\n===== Default rate por", c, "=====")
    result = spark.sql(value_counts_sql("application_train", c))
    result.show(30, truncate=False)

    pd_result = result.limit(15).toPandas()
    plot_bar_from_pandas(
        pd_result,
        x="valor",
        y="default_rate_pct",
        title="Tasa de default por {}".format(c),
        xlabel=c,
        ylabel="% default",
        rotation=45
    )


---
## 🔬 SECCIÓN 3 — EDA con SparkSQL sobre `bureau`


In [ ]:
# ============================================================
# CELDA 3.1 — Estructura y nulos de bureau
# ============================================================

sql("SELECT COUNT(*) AS total_filas FROM bureau")
sql("SELECT * FROM bureau LIMIT 10", rows=10, truncate=False)

nulls_bureau = spark.sql(build_null_sql("bureau", bureau.columns))
nulls_bureau.show(40, truncate=False)
guardar_spark_en_refined(nulls_bureau, "nulos_bureau")


In [ ]:
# ============================================================
# CELDA 3.2 — Créditos activos, cerrados y morosos
# ============================================================

credit_active = sql("""
SELECT
    credit_active,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bureau
GROUP BY credit_active
ORDER BY total DESC
""", rows=30, truncate=False)

guardar_spark_en_refined(credit_active, "bureau_credit_active")

credit_type = sql("""
SELECT
    credit_type,
    COUNT(*) AS total
FROM bureau
GROUP BY credit_type
ORDER BY total DESC
""", rows=50, truncate=False)

guardar_spark_en_refined(credit_type, "bureau_credit_type")


In [ ]:
# ============================================================
# CELDA 3.3 — Agregación bureau por cliente y join con TARGET
# ============================================================

bureau_agg = spark.sql("""
SELECT
    sk_id_curr,
    COUNT(*) AS bureau_num_records,
    SUM(CASE WHEN credit_active = 'Active' THEN 1 ELSE 0 END) AS bureau_active_credits,
    SUM(CASE WHEN credit_active = 'Closed' THEN 1 ELSE 0 END) AS bureau_closed_credits,
    SUM(CASE WHEN credit_active = 'Bad debt' THEN 1 ELSE 0 END) AS bureau_bad_debt_credits,
    ROUND(AVG(amt_credit_sum), 2) AS bureau_avg_credit_sum,
    ROUND(SUM(amt_credit_sum), 2) AS bureau_total_credit_sum,
    ROUND(SUM(amt_credit_sum_debt), 2) AS bureau_total_debt,
    MAX(credit_day_overdue) AS bureau_max_days_overdue,
    ROUND(AVG(days_credit), 2) AS bureau_avg_days_credit
FROM bureau
GROUP BY sk_id_curr
""")

bureau_agg.createOrReplaceTempView("bureau_agg")
bureau_agg.show(10, truncate=False)
guardar_spark_en_refined(bureau_agg, "bureau_agg")

bureau_target = sql("""
SELECT
    a.target,
    COUNT(*) AS total_clientes,
    ROUND(AVG(b.bureau_num_records), 2) AS avg_bureau_records,
    ROUND(AVG(b.bureau_total_debt), 2) AS avg_total_debt,
    ROUND(AVG(b.bureau_active_credits), 2) AS avg_active_credits
FROM application_train a
LEFT JOIN bureau_agg b
    ON a.sk_id_curr = b.sk_id_curr
WHERE a.target IN (0, 1)
GROUP BY a.target
ORDER BY a.target
""", rows=10, truncate=False)

guardar_spark_en_refined(bureau_target, "bureau_features_por_target")


---
## 🔬 SECCIÓN 4 — EDA con SparkSQL sobre `bureau_balance`


In [ ]:
# ============================================================
# CELDA 4.1 — Estados mensuales de bureau_balance
# ============================================================

sql("SELECT COUNT(*) AS total_filas FROM bureau_balance")

bb_status = sql("""
SELECT
    status,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM bureau_balance
GROUP BY status
ORDER BY total DESC
""", rows=20, truncate=False)

guardar_spark_en_refined(bb_status, "bureau_balance_status")

bb_agg = spark.sql("""
SELECT
    sk_id_bureau,
    COUNT(*) AS bb_num_months,
    SUM(CASE WHEN status = '0' THEN 1 ELSE 0 END) AS bb_status_0,
    SUM(CASE WHEN status = '1' THEN 1 ELSE 0 END) AS bb_status_1,
    SUM(CASE WHEN status IN ('2','3','4','5') THEN 1 ELSE 0 END) AS bb_late_months,
    SUM(CASE WHEN status = 'C' THEN 1 ELSE 0 END) AS bb_closed_months,
    SUM(CASE WHEN status = 'X' THEN 1 ELSE 0 END) AS bb_unknown_months
FROM bureau_balance
GROUP BY sk_id_bureau
""")

bb_agg.createOrReplaceTempView("bureau_balance_agg")
bb_agg.show(10, truncate=False)
guardar_spark_en_refined(bb_agg, "bureau_balance_agg")


---
## 🔬 SECCIÓN 5 — EDA con SparkSQL sobre `previous_application`


In [ ]:
# ============================================================
# CELDA 5.1 — Solicitudes previas: estado y tipo de contrato
# ============================================================

sql("SELECT COUNT(*) AS total_filas FROM previous_application")

prev_status = sql("""
SELECT
    name_contract_status,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
FROM previous_application
GROUP BY name_contract_status
ORDER BY total DESC
""", rows=30, truncate=False)

guardar_spark_en_refined(prev_status, "previous_application_status")

prev_contract = sql("""
SELECT
    name_contract_type,
    COUNT(*) AS total
FROM previous_application
GROUP BY name_contract_type
ORDER BY total DESC
""", rows=30, truncate=False)

guardar_spark_en_refined(prev_contract, "previous_application_contract_type")


In [ ]:
# ============================================================
# CELDA 5.2 — Agregación de solicitudes previas por cliente
# ============================================================

previous_agg = spark.sql("""
SELECT
    sk_id_curr,
    COUNT(*) AS prev_num_applications,
    SUM(CASE WHEN name_contract_status = 'Approved' THEN 1 ELSE 0 END) AS prev_approved,
    SUM(CASE WHEN name_contract_status = 'Refused' THEN 1 ELSE 0 END) AS prev_refused,
    ROUND(AVG(amt_application), 2) AS prev_avg_amt_application,
    ROUND(AVG(amt_credit), 2) AS prev_avg_amt_credit,
    ROUND(AVG(amt_down_payment), 2) AS prev_avg_down_payment
FROM previous_application
GROUP BY sk_id_curr
""")

previous_agg.createOrReplaceTempView("previous_application_agg")
previous_agg.show(10, truncate=False)
guardar_spark_en_refined(previous_agg, "previous_application_agg")

prev_target = sql("""
SELECT
    a.target,
    COUNT(*) AS total_clientes,
    ROUND(AVG(p.prev_num_applications), 2) AS avg_prev_applications,
    ROUND(AVG(p.prev_approved), 2) AS avg_prev_approved,
    ROUND(AVG(p.prev_refused), 2) AS avg_prev_refused
FROM application_train a
LEFT JOIN previous_application_agg p
    ON a.sk_id_curr = p.sk_id_curr
WHERE a.target IN (0, 1)
GROUP BY a.target
ORDER BY a.target
""", rows=10, truncate=False)

guardar_spark_en_refined(prev_target, "previous_features_por_target")


---
## 🔬 SECCIÓN 6 — EDA con SparkSQL sobre POS, tarjeta de crédito e installments


In [ ]:
# ============================================================
# CELDA 6.1 — POS_CASH_balance
# ============================================================

sql("SELECT COUNT(*) AS total_filas FROM pos_cash_balance")

pos_status = sql("""
SELECT
    name_contract_status,
    COUNT(*) AS total
FROM pos_cash_balance
GROUP BY name_contract_status
ORDER BY total DESC
""", rows=30, truncate=False)

guardar_spark_en_refined(pos_status, "pos_cash_status")

pos_agg = spark.sql("""
SELECT
    sk_id_curr,
    COUNT(*) AS pos_num_records,
    MAX(sk_dpd) AS pos_max_dpd,
    ROUND(AVG(sk_dpd), 2) AS pos_avg_dpd,
    MAX(sk_dpd_def) AS pos_max_dpd_def,
    ROUND(AVG(cnt_instalment_future), 2) AS pos_avg_installments_future
FROM pos_cash_balance
GROUP BY sk_id_curr
""")

pos_agg.createOrReplaceTempView("pos_cash_agg")
pos_agg.show(10, truncate=False)
guardar_spark_en_refined(pos_agg, "pos_cash_agg")


In [ ]:
# ============================================================
# CELDA 6.2 — credit_card_balance
# ============================================================

sql("SELECT COUNT(*) AS total_filas FROM credit_card_balance")

cc_status = sql("""
SELECT
    name_contract_status,
    COUNT(*) AS total
FROM credit_card_balance
GROUP BY name_contract_status
ORDER BY total DESC
""", rows=30, truncate=False)

guardar_spark_en_refined(cc_status, "credit_card_status")

cc_agg = spark.sql("""
SELECT
    sk_id_curr,
    COUNT(*) AS cc_num_records,
    ROUND(AVG(amt_balance), 2) AS cc_avg_balance,
    ROUND(MAX(amt_balance), 2) AS cc_max_balance,
    ROUND(AVG(amt_credit_limit_actual), 2) AS cc_avg_limit,
    MAX(sk_dpd) AS cc_max_dpd,
    MAX(sk_dpd_def) AS cc_max_dpd_def
FROM credit_card_balance
GROUP BY sk_id_curr
""")

cc_agg.createOrReplaceTempView("credit_card_agg")
cc_agg.show(10, truncate=False)
guardar_spark_en_refined(cc_agg, "credit_card_agg")


In [ ]:
# ============================================================
# CELDA 6.3 — installments_payments
# ============================================================

sql("SELECT COUNT(*) AS total_filas FROM installments_payments")

inst_agg = spark.sql("""
SELECT
    sk_id_curr,
    COUNT(*) AS inst_num_payments,
    ROUND(AVG(amt_instalment), 2) AS inst_avg_instalment,
    ROUND(AVG(amt_payment), 2) AS inst_avg_payment,
    ROUND(AVG(amt_payment - amt_instalment), 2) AS inst_avg_payment_diff,
    SUM(CASE WHEN days_entry_payment > days_instalment THEN 1 ELSE 0 END) AS inst_late_payments,
    ROUND(AVG(days_entry_payment - days_instalment), 2) AS inst_avg_days_delay
FROM installments_payments
GROUP BY sk_id_curr
""")

inst_agg.createOrReplaceTempView("installments_agg")
inst_agg.show(10, truncate=False)
guardar_spark_en_refined(inst_agg, "installments_agg")


---
## 🔗 SECCIÓN 7 — Integración de tablas y dataset analítico consolidado


In [ ]:
# ============================================================
# CELDA 7.1 — Crear features base desde application_train con SparkSQL
# ============================================================

app_features = spark.sql("""
SELECT
    *,
    ABS(days_birth) / 365.0 AS age_years,
    CASE WHEN days_employed = 365243 THEN 1 ELSE 0 END AS days_employed_anomaly,
    CASE WHEN days_employed = 365243 THEN NULL ELSE ABS(days_employed) / 365.0 END AS years_employed_clean,
    amt_credit / NULLIF(amt_income_total, 0) AS credit_income_ratio,
    amt_annuity / NULLIF(amt_income_total, 0) AS annuity_income_ratio,
    amt_goods_price / NULLIF(amt_credit, 0) AS goods_credit_ratio,
    CASE WHEN code_gender = 'XNA' THEN 'Unknown' ELSE code_gender END AS code_gender_clean
FROM application_train
WHERE target IN (0, 1)
""")

app_features.createOrReplaceTempView("application_train_features")
app_features.select("sk_id_curr", "target", "age_years", "credit_income_ratio", "annuity_income_ratio", "days_employed_anomaly", "code_gender_clean").show(10, truncate=False)


In [ ]:
# ============================================================
# CELDA 7.2 — Join de todas las features agregadas
# ============================================================

master_dataset = spark.sql("""
SELECT
    a.*,

    b.bureau_num_records,
    b.bureau_active_credits,
    b.bureau_closed_credits,
    b.bureau_bad_debt_credits,
    b.bureau_avg_credit_sum,
    b.bureau_total_credit_sum,
    b.bureau_total_debt,
    b.bureau_max_days_overdue,
    b.bureau_avg_days_credit,

    p.prev_num_applications,
    p.prev_approved,
    p.prev_refused,
    p.prev_avg_amt_application,
    p.prev_avg_amt_credit,
    p.prev_avg_down_payment,

    pos.pos_num_records,
    pos.pos_max_dpd,
    pos.pos_avg_dpd,
    pos.pos_max_dpd_def,
    pos.pos_avg_installments_future,

    cc.cc_num_records,
    cc.cc_avg_balance,
    cc.cc_max_balance,
    cc.cc_avg_limit,
    cc.cc_max_dpd,
    cc.cc_max_dpd_def,

    inst.inst_num_payments,
    inst.inst_avg_instalment,
    inst.inst_avg_payment,
    inst.inst_avg_payment_diff,
    inst.inst_late_payments,
    inst.inst_avg_days_delay

FROM application_train_features a
LEFT JOIN bureau_agg b
    ON a.sk_id_curr = b.sk_id_curr
LEFT JOIN previous_application_agg p
    ON a.sk_id_curr = p.sk_id_curr
LEFT JOIN pos_cash_agg pos
    ON a.sk_id_curr = pos.sk_id_curr
LEFT JOIN credit_card_agg cc
    ON a.sk_id_curr = cc.sk_id_curr
LEFT JOIN installments_agg inst
    ON a.sk_id_curr = inst.sk_id_curr
""")

master_dataset.createOrReplaceTempView("master_dataset")

print("Filas master:", master_dataset.count())
print("Columnas master:", len(master_dataset.columns))
master_dataset.select("sk_id_curr", "target", "age_years", "bureau_num_records", "prev_num_applications", "inst_late_payments").show(10, truncate=False)


In [ ]:
# ============================================================
# CELDA 7.3 — Validación del dataset maestro y guardado en trusted/gold
# ============================================================

sql("""
SELECT
    COUNT(*) AS total_filas,
    COUNT(DISTINCT sk_id_curr) AS clientes_unicos,
    SUM(CASE WHEN target IS NULL THEN 1 ELSE 0 END) AS nulos_target,
    SUM(CASE WHEN age_years IS NULL THEN 1 ELSE 0 END) AS nulos_age_years,
    SUM(CASE WHEN credit_income_ratio IS NULL THEN 1 ELSE 0 END) AS nulos_credit_income_ratio
FROM master_dataset
""")

# Guardar en trusted/gold. Descomenta cuando ya hayas validado rutas y permisos.
guardar_parquet_trusted(master_dataset, "application_train_prepared")


In [ ]:
# ============================================================
# CELDA 7.4 — Análisis cruzado: features nuevas vs TARGET
# ============================================================

master_target_summary = sql("""
SELECT
    target,
    COUNT(*) AS total,
    ROUND(AVG(age_years), 2) AS avg_age,
    ROUND(AVG(credit_income_ratio), 4) AS avg_credit_income_ratio,
    ROUND(AVG(annuity_income_ratio), 4) AS avg_annuity_income_ratio,
    ROUND(AVG(bureau_num_records), 2) AS avg_bureau_records,
    ROUND(AVG(prev_num_applications), 2) AS avg_prev_applications,
    ROUND(AVG(inst_late_payments), 2) AS avg_late_payments
FROM master_dataset
GROUP BY target
ORDER BY target
""", rows=10, truncate=False)

guardar_spark_en_refined(master_target_summary, "master_features_por_target")


---
## 📊 SECCIÓN 8 — Dashboard final de hallazgos


In [ ]:
# ============================================================
# CELDA 8.1 — Dashboard visual de hallazgos clave
# ============================================================

# 1. TARGET
p_target = spark.sql("""
SELECT target, COUNT(*) AS total
FROM master_dataset
GROUP BY target
ORDER BY target
""").toPandas()

# 2. Educación default rate
p_edu = spark.sql("""
SELECT
    name_education_type,
    ROUND(100.0 * SUM(target) / COUNT(*), 2) AS default_rate_pct,
    COUNT(*) AS total
FROM master_dataset
GROUP BY name_education_type
ORDER BY default_rate_pct DESC
""").toPandas()

# 3. Correlaciones
p_corr = spark.sql("""
SELECT variable, corr_target
FROM corr_target
ORDER BY ABS(corr_target) DESC
LIMIT 10
""").toPandas()

# 4. Nulos top 10
p_nulls = nulls_app.limit(10).toPandas()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].bar(p_target["target"].astype(str), p_target["total"])
axes[0, 0].set_title("Distribución de TARGET")
axes[0, 0].set_xlabel("TARGET")
axes[0, 0].set_ylabel("Cantidad")

axes[0, 1].bar(p_edu["name_education_type"].astype(str), p_edu["default_rate_pct"])
axes[0, 1].set_title("Default rate por educación")
axes[0, 1].set_ylabel("% default")
axes[0, 1].tick_params(axis="x", rotation=45)

axes[1, 0].barh(p_corr["variable"].astype(str), p_corr["corr_target"])
axes[1, 0].set_title("Top correlaciones con TARGET")
axes[1, 0].invert_yaxis()

axes[1, 1].barh(p_nulls["columna"].astype(str), p_nulls["pct_nulos"])
axes[1, 1].set_title("Top 10 columnas con nulos")
axes[1, 1].set_xlabel("% nulos")
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()


---
## 📝 SECCIÓN 9 — Hallazgos, decisiones y evidencias para el informe


In [ ]:
# ============================================================
# CELDA 9.1 — Tabla de decisiones para Track C
# ============================================================

decisiones = [
    ("TARGET desbalanceado", "Distribución de TARGET", "Usar métricas como AUC/recall/F1 en modelado", "Track D"),
    ("Nulos en variables financieras", "Reporte de nulos", "Imputar numéricas con mediana o crear flags de missing", "Track C"),
    ("DAYS_BIRTH negativo", "Análisis de variables DAYS", "Crear AGE_YEARS = ABS(days_birth)/365", "Track C"),
    ("DAYS_EMPLOYED con valor 365243", "Detección de anomalías", "Crear DAYS_EMPLOYED_ANOMALY y limpiar valor", "Track C"),
    ("CODE_GENDER contiene XNA", "Conteo categórico", "Reemplazar XNA por Unknown o moda", "Track C"),
    ("bureau tiene varias filas por cliente", "Agregación bureau", "Crear bureau_agg por sk_id_curr antes del join", "Track C"),
    ("previous_application tiene múltiples solicitudes por cliente", "Agregación previous", "Crear previous_application_agg por sk_id_curr", "Track C"),
    ("Datos preparados deben ser eficientes", "Arquitectura batch", "Guardar master_dataset en Parquet en trusted/gold", "Track C"),
    ("EDA debe dejar evidencia", "Resultados refinados", "Guardar tablas resumen en refined/silver", "Track B/C")
]

decisiones_df = spark.createDataFrame(decisiones, ["hallazgo", "evidencia_eda", "decision", "track"])
decisiones_df.show(truncate=False)
guardar_spark_en_refined(decisiones_df, "decisiones_track_c")


In [ ]:
# ============================================================
# CELDA 9.2 — Resumen ejecutivo automático
# ============================================================

resumen = """
RESUMEN EJECUTIVO DEL EDA

1. Los datos fueron leídos desde AWS Glue Data Catalog cuando las tablas catalogadas estuvieron disponibles.
2. El análisis exploratorio se realizó principalmente con SparkSQL mediante spark.sql().
3. Se analizó la variable objetivo TARGET y se identificó el nivel de desbalance de clases.
4. Se revisaron valores nulos en application_train y tablas auxiliares.
5. Se analizaron variables financieras, demográficas y scores externos.
6. Se detectaron valores anómalos como DAYS_EMPLOYED = 365243 y categorías especiales como XNA.
7. Se agregaron tablas auxiliares por sk_id_curr para evitar duplicar registros al hacer joins.
8. Se construyó un dataset maestro con features de application_train, bureau, previous_application, POS, tarjeta de crédito e installments.
9. El dataset consolidado se guardó en la zona trusted/gold en formato Parquet.
10. Los resultados del EDA se guardaron en refined/silver para documentación y visualización.
"""

print(resumen)

resumen_df = spark.createDataFrame([(resumen,)], ["resumen_ejecutivo"])
guardar_spark_en_refined(resumen_df, "resumen_ejecutivo_eda")


In [ ]:
# ============================================================
# CELDA 9.3 — Inventario de archivos generados en refined/silver
# ============================================================

print("Los resultados del EDA fueron enviados a:")
print(REFINED_PATH)
print("\nRevisa en S3 las carpetas generadas, por ejemplo:")
print("- inventario_raw/")
print("- target_distribution/")
print("- nulos_application_train/")
print("- correlaciones_target/")
print("- bureau_agg/")
print("- previous_application_agg/")
print("- master_features_por_target/")
print("- decisiones_track_c/")
print("- resumen_ejecutivo_eda/")


---
## ✅ SECCIÓN 10 — Cierre


In [ ]:
# ============================================================
# CELDA FINAL — Cierre opcional de SparkSession
# ============================================================

print("✅ Notebook EDA SparkSQL finalizado")
print("Recuerda cerrar la sesión interactiva de Glue si ya terminaste para evitar costos.")

# Descomenta solo si quieres cerrar Spark completamente.
# spark.stop()
